In [8]:
import lxml.html
import requests
import time
import uuid
import pandas as pd

In [10]:
def make_request(url):
    user_agent = "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36"
    acc = "text/html,application/xhtml+xml,application/xml;q=0.9,*/*;q=0.8"
    headers = {
        "User-Agent": user_agent,
        "Accept": acc,
        "Accept-Language": "en-US,en;q=0.5",
        "Accept-Encoding": "gzip, deflate",
        "Connection": "keep-alive",
    }

    resp = requests.get(url, headers=headers)

    if resp.status_code == 429:
        time.sleep(2.5)
        resp = make_request(url)
    elif resp.status_code == 200:
        return resp

In [11]:
def scrape_judge(url):
    j_page = lxml.html.fromstring(make_request(url).text)
    r_d = {}

    xp1 = "//div[@class = 'judge-info']//div[@class = 'about-list']"
    xp2 = "//div[@class = 'about-list-item']//h3/text()"
    item_titles = j_page.xpath(xp1 + xp2)

    xp1 = "//div[@class = 'judge-info']//div[@class = 'about-list']"
    xp2 = "//div[@class = 'about-list-item']//p/text()"
    items = j_page.xpath(xp1 + xp2)

    for i, title in enumerate(item_titles):
        r_d[title.lower()] = items[i].replace("\t", "").strip().lower()

    return r_d

In [16]:
def scrape_main(url):
    main_page = lxml.html.fromstring(make_request(url).text)

    xp1 = "//section[@filter = 'judge']//"
    xp2 = "a[contains(@href, 'judge')]//@href"
    judge_links = main_page.xpath(xp1 + xp2)

    xp1 = "//section[@filter = 'judge']//a[contains(@href, 'judge')]//"
    xp2 = "div[@class = 'module--content module--content-post']"
    xp3 = "//h2[@class = 'title']/text()"
    judge_names = main_page.xpath(xp1 + xp2 + xp3)

    xp1 = "//div[@class = 'about-icons']"
    xp2 = "//div[@class = 'about-icon' and @data-type = 'state']"
    xp3 = "//span[not(contains(@class, 'sr-only'))]/text()"
    judge_states = main_page.xpath(xp1 + xp2 + xp3)

    judge_fields = [
        "gender",
        "party",
        "race",
        "professional experience",
        "election type",
        "term start",
        "term end",
        "next election date",
    ]

    judge_pd = {"JID": [], "name": [], "state": []}

    for field in judge_fields:
        judge_pd[field] = []

    for i, name in enumerate(judge_names):
        judge_pd["name"].append(name)
        judge_pd["JID"].append(uuid.uuid4())
        judge_pd["state"].append(judge_states[i])

        judge_dic = scrape_judge(judge_links[i])
        print(f"Scraped judge: {name}")

        if judge_dic == {}:
            break

        for field in judge_fields:
            if field not in judge_dic:
                data = pd.NA
            else:
                data = judge_dic[field]

            judge_pd[field].append(data)

    return pd.DataFrame(judge_pd)

In [17]:
url = "https://state-law-research.org/state-justices/"

judge_pd = scrape_main(url)
judge_pd

Scraped judge: Abigail LeGrow
Scraped judge: Adam Tanenbaum
Scraped judge: Aimee A. Oravec
Scraped judge: Alisa Kelli Wise
Scraped judge: Allison Riggs
Scraped judge: Andrew J. McDonald
Scraped judge: Andrew M. Horton
Scraped judge: Andrew M. Mead
Scraped judge: Andrew Pinson
Scraped judge: Angela M. Eaves
Scraped judge: Angela McCormick Bisig
Scraped judge: Anita Earls
Scraped judge: Ann A. Scott Timmer
Scraped judge: Anna Blackburne-Rigsby
Scraped judge: Anne K. McKeig
Scraped judge: Anne M. Patterson
Scraped judge: Annette Kingsland
  Ziegler
Scraped judge: Anthony Cannataro
Scraped judge: Aruna Masih
Scraped judge: Barbara A. Madsen
Scraped judge: Barbara Webb
Scraped judge: Benjamin A. Land
Scraped judge: Bert Richardson
Scraped judge: Beth Baker
Scraped judge: Brady E. Mendheim, Jr.
Scraped judge: Brett Busby
Scraped judge: Brian D. Boatright
Scraped judge: Brian Hagedorn
Scraped judge: Brian K. Zahra
Scraped judge: Briana Zamora
Scraped judge: Bronson James
Scraped judge: Bryan 

,JID,name,state,gender,party,race,professional experience,election type,term start,term end,next election date
0,eb887c0e-f844-484a-bb86-42442231e877,Abigail LeGrow,DE,female,unsure,white,"corporate, judicial","appointed, leg confirmed","may 3, 2023",2035,requires re-nomination and re-confirmation
1,3e411aa7-15d0-41ba-b822-c977aeb3407a,Adam Tanenbaum,FL,male,republican,white,"government civil (non civil rights), judicial,...",appointed,"january 14, 2026","january 2, 2029",<NA>
2,0e46adc7-982e-48f2-ba90-8a5f7739707c,Aimee A. Oravec,AK,female,unsure,white,"corporate, misc. private practice",appointed,"january 31, 2025",2028,2028
3,74cf0c52-33b0-4fa8-84dc-3cd15f1a62b2,Alisa Kelli Wise,AL,female,republican,white,"corporate, misc. private practice, judicial, o...","elected, partisan",2011,"january 15, 2029","november 2, 2028"
4,8791a09e-321b-4a30-95d8-56588932b423,Allison Riggs,NC,female,democrat,white,"legal aid/non-govt civil rights, judicial",appointed,"september 11, 2023","december 31, 2024","november 7, 2024"
...,...,...,...,...,...,...,...,...,...,...,...
340,c7fb3196-9702-4565-abcf-eaa21a0869e2,"William J. Musseman, Jr.",OK,male,republican,white,"prosecutor, judicial",appointed,2022,unsure,unsure
341,200a7b12-a9c1-4c66-a275-b50f39696f89,William P. Robinson III,RI,male,republican,white,misc. private practice,"appointed, leg confirmed",2004,lifetime,n/a - appointed for life
342,95044e43-7ecd-42fc-a881-ec227f247d47,William R. Wooton,WV,male,democrat,white,"prosecutor, plaintiff-side civil private pract...","elected, nonpartisan","january 1, 2021",december 2032,tbd
343,81f897ec-21e1-4222-857f-4c2afb5c173f,"William W. Hood, III",CO,male,democrat,white,"prosecutor, corporate, plaintiff-side civil pr...","appointed, retention elected","january 13, 2014","january 11, 2027","november 3, 2026"


In [22]:
judge_pd[judge_pd["state"] == "LA"]

,JID,name,state,gender,party,race,professional experience,election type,term start,term end,next election date
34,fb3b4e4d-083f-43df-8c80-f0aa88a7fc9b,Cade Cole,LA,male,republican,white,"prosecutor, government civil (non civil rights...","elected, partisan",march 2025,"december 31, 2026","november 3, 2026"
139,60091a4a-11cd-47fa-a338-a2394de3878f,Jay B. McCallum,LA,male,republican,white,"prosecutor, public defender/other indigent def...","elected, partisan",november 2020,"december 31, 2026","november 3, 2026"
140,895161fe-028f-4481-8abd-d598cb0a761f,Jefferson D. Hughes III,LA,male,republican,white,"misc. private practice, judicial","elected, partisan",february 2013,"december 31, 2028","november 7, 2028"
155,48bacdca-711a-4830-86fc-9c00a180f65a,John Guidry,LA,male,democrat,black,"judicial, legislative experience","elected, partisan","january 1, 2025",<NA>,<NA>
158,c2f8d79a-2170-4d5e-8d7f-1af544a44d99,John L. Weimer,LA,male,independent,white,"misc. private practice, judicial, law professo...","elected, partisan",2001,"december 31, 2032","november 2, 2032"
261,2e1df48f-ebed-43b5-b5c1-2d86d1c7dc71,Piper D. Griffin,LA,female,democrat,black,"misc. private practice, judicial","elected, partisan","january 1, 2021","december 31, 2030","november 5, 2030"
